In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)

# -----------------------------
# 1. GENERATION DES CLIENTS
# -----------------------------
N_CUSTOMERS = 2000
N_TRANSACTIONS = 10000

countries = ["FR","FR","FR","DE","ES","IT","GB","US","CA","MA","DZ","TN"]
country_risk_map = {"FR":0.1,"DE":0.1,"ES":0.15,"IT":0.2,"GB":0.2,"US":0.3,"CA":0.1,
                    "MA":0.4,"DZ":0.4,"TN":0.35}

customer_id = np.arange(1, N_CUSTOMERS+1)
age = np.random.randint(18, 80, N_CUSTOMERS)
account_age_years = np.round(np.random.exponential(scale=5, size=N_CUSTOMERS), 1)
income = np.round(np.random.normal(30000, 15000, N_CUSTOMERS), 2)
income = np.clip(income, 8000, 200000)

customer_country = np.random.choice(countries, N_CUSTOMERS)
risk_segment = np.where(income < 15000, "high",
                 np.where(income < 40000, "medium", "low"))

avg_transaction_amount = np.round(np.random.exponential(scale=60, size=N_CUSTOMERS), 2)
preferred_device_risk = np.clip(np.random.normal(0.2, 0.1, N_CUSTOMERS), 0, 1)
fraud_history = np.random.choice([0,1], N_CUSTOMERS, p=[0.95, 0.05])

customers = pd.DataFrame({
    "customer_id": customer_id,
    "age": age,
    "account_age_years": account_age_years,
    "country": customer_country,
    "income": income,
    "risk_segment": risk_segment,
    "avg_transaction_amount": avg_transaction_amount,
    "preferred_device_risk": preferred_device_risk,
    "fraud_history": fraud_history
})

customers.to_csv("customers.csv", index=False)
print("customers.csv généré.")


# -----------------------------
# 2. GENERATION DES TRANSACTIONS
# -----------------------------
bins = [
    ("4539", "Visa", "FR", 0.1),
    ("4929", "Visa", "FR", 0.1),
    ("5100", "Mastercard", "FR", 0.15),
    ("5200", "Mastercard", "DE", 0.2),
    ("3792", "Amex", "US", 0.3),
    ("6011", "Discover", "US", 0.4)
]

# Sélection aléatoire de clients
tx_customer = np.random.choice(customer_id, N_TRANSACTIONS)

# Montants dépendant du profil client
amount = []
for cid in tx_customer:
    base = customers.loc[cid-1, "avg_transaction_amount"]
    amt = np.random.exponential(scale=base)
    amount.append(amt)
amount = np.round(amount, 2)

# BIN
bin_idx = np.random.choice(len(bins), N_TRANSACTIONS)
bin_code = [bins[i][0] for i in bin_idx]
card_type = [bins[i][1] for i in bin_idx]
issuer_country = [bins[i][2] for i in bin_idx]
issuer_risk = [bins[i][3] for i in bin_idx]

# Pays de transaction
transaction_country = np.random.choice(countries, N_TRANSACTIONS)
country_risk = [country_risk_map[c] for c in transaction_country]

# Device
device_risk = np.clip(np.random.normal(0.25, 0.15, N_TRANSACTIONS), 0, 1)
new_device = np.random.choice([0,1], N_TRANSACTIONS, p=[0.92, 0.08])

# Geo anomaly
geo_anomaly = np.random.choice([0,1], N_TRANSACTIONS, p=[0.97, 0.03])

# Velocity
velocity_24h = np.random.poisson(2, N_TRANSACTIONS)
velocity_risk = (velocity_24h > 6).astype(int)

# Heure
hour = np.random.randint(0, 24, N_TRANSACTIONS)
night_risk = ((hour < 5)).astype(int)

# Fraude historique client
customer_fraud_history = customers.set_index("customer_id").loc[tx_customer, "fraud_history"].values

# Score de fraude réaliste
fraud_score = (
    0.30 * (amount > 300).astype(int) +
    0.20 * np.array(country_risk) +
    0.15 * device_risk +
    0.15 * velocity_risk +
    0.20 * geo_anomaly +
    0.10 * new_device +
    0.10 * night_risk +
    0.10 * customer_fraud_history
)

is_fraud = (fraud_score > 0.75).astype(int)

transactions = pd.DataFrame({
    "transaction_id": np.arange(1, N_TRANSACTIONS+1),
    "customer_id": tx_customer,
    "bin": bin_code,
    "card_type": card_type,
    "issuer_country": issuer_country,
    "issuer_risk": issuer_risk,
    "amount": amount,
    "transaction_country": transaction_country,
    "country_risk": country_risk,
    "device_risk": np.round(device_risk, 2),
    "new_device": new_device,
    "geo_anomaly": geo_anomaly,
    "hour": hour,
    "velocity_24h": velocity_24h,
    "is_fraud": is_fraud
})

transactions.to_csv("transactions.csv", index=False)
print("transactions.csv généré (10k transactions bancaires réalistes).")


customers.csv généré.
transactions.csv généré (10k transactions bancaires réalistes).
